# RAG Pipeline — Apple & Google 10-Ks

End-to-end pipeline: **PDF → Markdown → Chunks → Embeddings/Index → Retrieval**.

Each stage below just calls the existing pipeline modules (`pdf_to_markdown`, `chunking`, `embeddings`, `retreiver`, `rag_chat`) — this notebook is orchestration only, no reimplemented logic. Run top to bottom once; after that, just re-run the query cells at the bottom as many times as you like — the embedding and cross-encoder models stay loaded in memory instead of reloading on every call like the CLI scripts do.

Stops at retrieval — no chat/LLM step yet.

In [1]:
from pathlib import Path

import pandas as pd

import pdf_to_markdown
import chunking
import embeddings
import retreiver
from rag_chat import retrieve_multi, answer_query

DOCUMENTS_FOLDER = Path("documents")

/Users/sarthakkumar/Desktop/RAG/venv311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6256.42it/s]


## 1. PDF → Markdown

Converts every PDF in `documents/pdf/` to markdown (`documents/markdown/`) and extracts figures to `documents/images/`, using docling's layout + OCR models. This reprocesses every PDF each run (not just new ones), so expect a couple of minutes per document.

In [ ]:
pdf_to_markdown.process_folder(str(DOCUMENTS_FOLDER))

## 2. Markdown → Chunks

Parses each markdown file into heading / paragraph / table / image blocks and chunks them, merging short fragments (e.g. signature-block lines) so they don't end up as standalone, context-free chunks. Output goes to `documents/chunks/`.

In [ ]:
chunking.process_folder(str(DOCUMENTS_FOLDER))

## 3. Chunks → Embeddings + FAISS Index

Embeds every chunk with the local `all-MiniLM-L12-v2` model and builds a single FAISS index across *all* documents — chunks from different companies stay distinguishable via the `document` metadata field, so adding a third company later is just: drop its PDF into `documents/pdf/` and re-run stages 1–3. Output goes to `documents/embeddings/` and `documents/index/`.

In [ ]:
embeddings.process_folder(str(DOCUMENTS_FOLDER))

## 4. Retrieval

Two-stage: cosine similarity narrows the field, then a cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) reranks the survivors by scoring the actual (query, passage) pair jointly — much better at catching relevance a raw embedding average would miss (e.g. a term buried in a longer paragraph).

The first call below loads both models into memory; that only happens once per kernel session, so every query cell after this stays fast.

In [3]:
def show_results(result):
    rows = [
        {
            "score": round(r["final_score"], 3),
            "cosine": round(r["cosine"], 3),
            "document": r["metadata"]["document"],
            "heading": r["metadata"]["heading"],
            "text": r["text"][:200],
        }
        for r in result["results"]
    ]
    return pd.DataFrame(rows)


query = "what is artificial intelligence"
result = retreiver.retrieve(DOCUMENTS_FOLDER, query, meta_filter={})
show_results(result)

,score,cosine,document,heading,text
0,0.485,0.477,GOOGLE_FINANCE,"We are subject to a variety of new, existing, ...","approaches to AI regulation. For instance, in ..."
1,0.261,0.354,GOOGLE_FINANCE,Overview,As our founders Larry and Sergey wrote in the ...
2,0.014,0.369,APPLE_FINANCE,"The Company's business, results of operations ...",The Company has faced and continues to face a ...


## 5. Multi-document queries

A single global top-3 can let one document's higher-scoring chunks crowd another out entirely — e.g. a query naming two companies could silently return results for only one. `retrieve_multi` (from `rag_chat.py`) detects every document mentioned by name and retrieves each separately before merging, so every mentioned document is guaranteed representation.

In [4]:
query = "what is googles revenue in 2024 and what is apple revenue in 2024"
result = retrieve_multi(DOCUMENTS_FOLDER, query, meta_filter={}, heading=None, subheading=None)
print("matched documents:", result.get("matched_documents"))
show_results(result)

matched documents: ['APPLE_FINANCE', 'GOOGLE_FINANCE']


,score,cosine,document,heading,text
0,0.943,0.488,APPLE_FINANCE,Note 2 - Revenue,The Company's proportion of net sales by...
1,0.939,0.544,APPLE_FINANCE,Note 2 - Revenue,"As of September 27, 2025 and September 28, 202..."
2,0.737,0.548,APPLE_FINANCE,"The technology industry, including, in some in...","Further, the Company has commercial relat..."
3,0.999,0.672,GOOGLE_FINANCE,"Google Subscriptions, Platforms, and Devices","Google subscriptions, platforms, and devices r..."
4,0.999,0.654,GOOGLE_FINANCE,Google Search & other,Google Search & other revenues increased $26.4...
5,0.998,0.577,GOOGLE_FINANCE,Google Network,Google Network revenues decreased $567 million...


## Try your own query

Edit `query` below and re-run — no need to redo the earlier stages.

In [ ]:
query = "your question here"
result = retrieve_multi(DOCUMENTS_FOLDER, query, meta_filter={}, heading=None, subheading=None)
show_results(result)

## 6. Chat

Ties everything together: each question runs through `answer_query` (retrieval via `retrieve_multi` + prompt building + a local Llama 3.1 8B call through Ollama), and the answer is grounded only in the retrieved excerpts. Each question is answered independently - there's no memory of earlier turns in this loop, so follow-up questions need to be self-contained (e.g. say "Google" again rather than "they").

Run the cell, type a question, and keep going. Type `end` to stop.

In [5]:
CHAT_MODEL = "llama3.1:8b"

print("Ask a question about the loaded documents. Type 'end' to stop.\n")
while True:
    query = input("You: ").strip()
    if query.lower() == "end":
        print("Chat ended.")
        break
    if not query:
        continue

    result = answer_query(DOCUMENTS_FOLDER, query, model=CHAT_MODEL)

    sources = result["retrieval"].get("results", [])
    used_docs = sorted({s["metadata"]["document"] for s in sources})

    print(f"[sources: {', '.join(used_docs) if used_docs else 'none'}]")
    print(f"Assistant: {result['answer']}\n")

Ask a question about the loaded documents. Type 'end' to stop.

[sources: APPLE_FINANCE, GOOGLE_FINANCE]
Assistant: Based on the provided sources, I can answer the following:

* Google's revenue in 2024: 
According to Source 5, Google Search & other revenues increased $26.4 billion from 2024 to 2025. This implies that Google's revenue in 2024 was $26.4 billion less than in 2025.
However, we don't have a direct figure for Google's revenue in 2024.

* Apple's revenue in 2024: 
According to Source 2, as of September 28, 2024, the Company (Apple) had total deferred revenue of $12.8 billion. However, this is not the same as the company's actual revenue.
We don't have a direct figure for Apple's revenue in 2024.

I don't know based on the given documents.

Chat ended.
